# wim1: growth of wind waves in a single column with wim_simple

The wind blows for half a day over one ocean cell. The cell is periodic in $x$ and $y$, so there is no spatial gradient and no propagation in space. The experiment shows how the wave spectrum grows from a small seed under a constant wind.

wim_simple solves the action balance $\partial_t A + \dot{\mathbf x}\cdot\nabla_{\mathbf x} A + \dot{\mathbf k}\cdot\nabla_{\mathbf k} A = S_{in} + S_{out} + S_{nl}$ (doc/wim_simple.pdf). In this column the transport terms vanish. What is left is the local balance of the energy density $E = \sigma A$,

$$\partial_t E = \gamma_{in} E + \gamma_{out} E + \dot E_{nl}.$$

$\gamma_{in}$ is the WAMDI wind input. $\gamma_{out}$ is the WAMDI dissipation by wave breaking. $\dot E_{nl}$ is the nonlinear transfer by the discrete interaction approximation (DIA) of WAM.

$E(\theta,\nu)$ is the variance density of the surface elevation per frequency $\nu$ and direction $\theta$. Its unit is m$^2$ s. The significant wave height is $H_s = 4\sqrt{\int E\, d\theta\, d\nu}$.

The direction $\theta$ is the direction the waves travel to. The direction vector is $\mathbf s = (\sin\theta, \cos\theta)$. $\theta = 0$ points to $+y$, and $\theta$ increases clockwise.

In [ ]:
import sys; sys.path.append('../..')
import pyOM3 as OM
from pyOM3.wim_simple.model import model
import numpy as np
import pylab as plt
from IPython import display

## Diagnostics

The class `my_model` adds a figure to the model. Its method `diagnose` is called by `loop` after every time step. Every 20 minutes of model time it redraws six panels.

`spatial_mean` averages over the real cells, and `angle_mean` averages over the real direction bins. Both cut off the halo points.

1. **Spectrum $E(\theta,\nu)$** on a polar plot, with direction as angle and radian frequency $\omega$ as radius. The plot follows the model convention, with $\theta = 0$ at the top ($+y$) and $\theta$ increasing clockwise, so it reads like a compass rose of the direction the waves travel to.
2. **Directional mean** $\frac{1}{2\pi}\int E\, d\theta$ as a function of $\omega$.
3. **Growth rates.** The directional mean of $\gamma_{in}$ is shown in blue. The negative dissipation rate $-\gamma_{out}$ is shown in red. $\gamma_{out}$ does not depend on direction.
4. **Source terms** of the energy, averaged over direction. $\dot E_{nl}$ is blue, $\gamma_{out}E$ is red and $\gamma_{in}E$ is black. Their unit is m$^2$.
5. **Stokes velocity** $u_S$ (blue) and $v_S$ (red) from `calc_stokes`. It is evaluated on an ocean column of 150 cells with $\Delta z = 0.1$ m, i.e. down to 15 m. Each cell gets the average over the cell, including the analytic $\nu^{-5}$ tail.
6. **Stress force** $\partial_z \boldsymbol\tau$ from `calc_stress_work`, on the same column. It is the momentum the waves lose by breaking and give to the ocean at depth $z$, in m/s$^2$. The $x$ component is blue and the $y$ component red. It is plotted in $10^{-3}$ m/s$^2$. The title shows the surface stress $\boldsymbol\tau_o$ of `calc_stress` in $10^{-3}$ m$^2$/s$^2$. The sum of panel 6 times $\Delta z$ equals $\boldsymbol\tau_o$ when the column is deep enough.

`P` is computed but not plotted.

In [ ]:
class my_model(model):
    
      def __init__(self,p):
          super().__init__(p)
          self.fig = plt.figure(figsize=(4*6,3))
          return
         
      def diagnose(self,n,tau): 
          
          def spatial_mean(a):
             A = self.nx*self.dx*self.ny*self.dy 
             b = a[...,self.halo:-self.halo,self.halo:-self.halo] 
             return np.sum(np.sum(b*self.dx,axis=-1)*self.dy,axis=-1)/A
              
          def angle_mean(a):
              b = a[self.halo:-self.halo,...]
              return np.sum( b*self.dd,axis=0)/(2*OM.np.pi)
              
          t =  n*self.dt
          #if t>86400: self.udir = self.udir*0 +np.pi/2
          if n==0 or OM.np.mod(t,60*20) == 0: # plotting
             
             Snonlin,_ =  self.Snonlin(self.E)
             Snonlin = angle_mean(  spatial_mean(  Snonlin ) )
             dz, nz = 0.1 ,150
             z = OM.np.arange(nz)*dz 
             z = z - z[-1] - dz/2   
             ust     = self.calc_stokes(dz,nz)[0,:,0,0]
             vst     = self.calc_stokes(dz,nz)[1,:,0,0]
             taux    = self.calc_stress_work(dz,nz)[0,:,0,0]
             tauy    = self.calc_stress_work(dz,nz)[1,:,0,0] 
             staux    = self.calc_stress()[0,0,0]
             stauy    = self.calc_stress()[1,0,0]  
             S_diss = angle_mean( spatial_mean( self.gamma_diss[None,:,:,:]*self.E )) 
             S_in   = angle_mean( spatial_mean( self.gamma_in*self.E ))  
             gamma_diss = spatial_mean( self.gamma_diss ) 
             gamma_in = angle_mean(spatial_mean(self.gamma_in))
             E = spatial_mean(self.E) 
             P = spatial_mean( 2*self.E* self.omega_t[None,:,None,None]**3/self.grav )
              
             self.fig.clf()               
             ax = self.fig.add_subplot(1,6,1, projection='polar') 
             ax.set_theta_zero_location('N')   # theta=0 towards +y
             ax.set_theta_direction(-1)        # theta increases clockwise
             co=plt.pcolormesh(self.theta[1:-1],self.omega_t[1:-1]
                            ,E[1:-1,1:-1].transpose(),cmap='jet')
             plt.colorbar(co) 
             plt.title('E at t=%6.1fd $[m^2 s]$'%(t/86400,))
             #plt.xlabel('$\\omega$ [1/s]')
             ax = self.fig.add_subplot(1,6,2)   
             plt.plot(self.omega_t[1:-1], angle_mean(E)[1:-1] )
              
             ax = self.fig.add_subplot(1,6,3)   
             plt.plot(self.omega_t[1:-1],gamma_in[1:-1],'*-')
             plt.plot(self.omega_t[1:-1],-gamma_diss[1:-1],'r*-')  
             plt.title('mean growth rates [1/s]') 
             plt.xlabel('$\\omega$ [1/s]') 
             ax = self.fig.add_subplot(1,6,4)   
             plt.plot(self.omega_t[1:-1],Snonlin[1:-1],'*-')
             plt.plot(self.omega_t[1:-1],S_diss[1:-1],'r*-')
             plt.plot(self.omega_t[1:-1],S_in[1:-1],'k*-') 
             plt.title('$S_{nonlin}$, $S_{in}$, $S_{diss}$ $[m^2]$ ')
             plt.xlabel('$\\omega$ [1/s]') 
             ax = self.fig.add_subplot(1,6,5)   
             plt.plot(ust,z,vst,z,'r') 
             plt.title('Stokes velocity [m/s]')
             plt.ylabel('z [m]') 
             ax = self.fig.add_subplot(1,6,6)   
             plt.plot(taux*1e3,z,tauy*1e3,z,'r') 
             
             plt.title('$\\partial_z\\tau$ $[10^{-3} m/s^2]$, $\\tau_o$=%.4f %.4f $[10^{-3} m^2/s^2]$'%(staux*1e3,stauy*1e3))
             plt.ylabel('z [m]') 
 
             display.clear_output(wait=True)  
             display.display(self.fig)
             

## Parameters

| key | value | meaning |
|---|---|---|
| `nx`, `ny` | 1, 1 | one cell |
| `dx`, `dy` | 1 km | cell size. It does not matter here, since there are no gradients |
| `periodic_in_x`, `periodic_in_y` | True | the cell is its own neighbour, so no coast and no gradient |
| `nf`, `f0`, `df0` | 36, 0.035 Hz, 0.1 | geometric frequency grid $\nu_{n+1} = 1.1\,\nu_n$ from 0.035 Hz to 0.98 Hz (the WAM grid) |
| `nd` | 40 | direction bins of 9 degrees |
| `dt` | 600 s | time step |
| `snonlin_backend` | `'fortran'` | the DIA from the original WAM code. `'numpy'` gives the same result to round-off |
| `mask_energy` | True | prognostic cut-off $\nu_c = \max(2.5\bar\nu, 3\nu_{pm})$ per column. Above it the spectrum is the imposed $\nu^{-5}$ tail |
| `limit_growth` | True | WAM growth limiter. It caps the source per time step |
| `implicit_dissipation_only` | True | only the dissipative part of the source feedback goes into the implicit denominator. Growth is explicit and bounded by the limiter |

The last three options are the WAM construction of the source step. They are described in doc/wim_simple.pdf, sections "High frequency cut-off" and "Time stepping scheme".

In [ ]:
p = {'nx':1,'ny':1,'nf':36,'nd':40,'dx':1e3,'dy':1e3,'dt':10*60.,
     'f0':0.035,'df0':0.1,'periodic_in_x':True,'periodic_in_y':True,
     'mask_energy':True,'limit_growth':True,
     'snonlin_backend':'fortran','implicit_dissipation_only':True}

## Initial state, forcing and run

`setup` builds the grid and allocates the fields. The arrays have the layout `(Nd, Nf, Ny, Nx)` with one halo point on each side, so `[1:-1]` selects the real bins and cells.

The seed is $E = 10^{-2}$ m$^2$ s in every real bin. It is white in frequency and isotropic, with $H_s \approx 1.0$ m. The action follows as $A = E/\sigma$, since `A` is the prognostic variable.

The friction velocity is $u_\star = 0.35$ m/s everywhere. The wind direction is `udir` $= \pi$, so the waves grow towards $-y$. The input is active only where $28\,(u_\star/c)\cos(\theta - \theta_\star) > 1$, with phase speed $c = g/\sigma$. The fully developed peak lies near $\nu_{pm} = g/(28 \cdot 2\pi u_\star) = 0.16$ Hz.

`loop` runs half a day, i.e. 72 steps of 600 s. Each step propagates the action (here without effect) and then applies the sources with the semi-implicit update. `loop` calls `diagnose` after every step. The interrupt button stops the run cleanly. The setup and the loop stay in one cell. Jupyter closes the figure of `my_model` at the end of the cell that created it, and `plt` would then draw into a new figure.

In [ ]:
M = my_model(p)
M.setup()
M.E[1:-1,1:-1,1:-1,1:-1] = 1e-2
M.A = M.E/M.omega_t[None,:,None,None]
M.ustar = OM.np.ones( (M.Ny,M.Nx) , OM.prec)*0.35
M.udir  = OM.np.ones( (M.Ny,M.Nx) , OM.prec) *np.pi

try:
    M.loop(int(86400*0.5/M.dt))
except KeyboardInterrupt:
    display.clear_output()

## What to expect

The numbers below are from a run with this configuration and `snonlin_backend='numpy'`.

| t [h] | $H_s$ [m] | peak $\nu$ [Hz] | $\tau_o^y$ [m$^2$/s$^2$] | $v_S$ at the top [m/s] |
|---|---|---|---|---|
| 2 | 0.94 | 0.31 | $-2.8\cdot10^{-6}$ | $-0.04$ |
| 4 | 1.08 | 0.29 | $-9.7\cdot10^{-6}$ | $-0.08$ |
| 6 | 1.22 | 0.24 | $-1.8\cdot10^{-5}$ | $-0.10$ |
| 8 | 1.36 | 0.21 | $-2.2\cdot10^{-5}$ | $-0.11$ |
| 10 | 1.48 | 0.20 | $-2.6\cdot10^{-5}$ | $-0.12$ |
| 12 | 1.57 | 0.20 | $-2.7\cdot10^{-5}$ | $-0.12$ |

First $H_s$ drops. The seed loses the part that travels against the wind and the high frequencies to dissipation. Then the wind sea grows along $-y$ and its peak moves to lower frequency. After 12 hours the peak is still above $\nu_{pm} = 0.16$ Hz, so the sea is still young.

The stress on the ocean grows with the sea. At 12 hours the wind puts $7.0\cdot10^{-5}$ m$^2$/s$^2$ of momentum into the resolved spectrum. Less than half of it reaches the ocean, the rest stays in the growing waves.

The $x$ components of $\boldsymbol\tau_o$ and $\mathbf u_S$ are zero to round-off, since wind and seed are symmetric about the $y$ axis.